In [ ]:
!pip install openai
!pip install pandas
!pip install matplotlib
!pip install seaborn
!pip install ipywidgets
!pip install reportlab



In [ ]:
from openai import OpenAI


API_KEY = "361a27a60f6b4138982fd15278917fed"

client = OpenAI(
    base_url="https://api.aimlapi.com/v1",
    api_key=API_KEY,   
)

try:
    response = client.chat.completions.create(
        model="openai/gpt-5-chat-latest",
        messages=[
            {"role": "user", "content": "Hello, tell me the capital of pakistan and 3 places to visit there"}
        ],
        temperature=0.7,
        top_p=0.7,
        frequency_penalty=1,
        max_tokens=200,   # correct parameter name
    )

    print("Assistant:", response.choices[0].message.content)

except Exception as e:
    print("Error:", e)

In [ ]:
# --- Setup ---
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openai import OpenAI
from google.colab import files

#  API 
client = OpenAI(
    base_url="https://api.aimlapi.com/v1",
    api_key="361a27a60f6b4138982fd15278917fed"
)


uploaded = files.upload()
dataframes = {}

if uploaded:
    for fname in uploaded.keys():
        ext = fname.split(".")[-1].lower()
        try:
            if ext == "csv":
                df = pd.read_csv(fname)
            elif ext in ["xls", "xlsx"]:
                df = pd.read_excel(fname)
            elif ext == "json":
                df = pd.read_json(fname)
            else:
                print(f"⚠️ Unsupported file type: {fname}")
                continue
            dataframes[fname] = df
            print(f"✅ File '{fname}' uploaded successfully with shape {df.shape}")
        except Exception as e:
            print(f"❌ Error reading {fname}: {e}")
else:
    print("⚠️ No files uploaded")


if dataframes:
    fname, df = list(dataframes.items())[0]
    dataset_summary = f"""
    Dataset: {fname}
    Shape: {df.shape}
    Columns: {df.dtypes.to_dict()}
    Head:\n{df.head(3).to_string()}
    """
else:
    dataset_summary = "No dataset available."
    df = None






chat_history = [
    {"role": "system", "content": "You are a professional data scientist. \
     When asked, suggest the best possible visualizations for the dataset summary \
     (scatter plots, histograms, line plots, bar plots, heatmaps, etc). \
     Wait for the user to pick one, then the code will generate it."},
    {"role": "user", "content": f"Here is my dataset summary:\n{dataset_summary}"}
]

print("\n You can now chat with your Data Scientist Assistant! Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("👋 Ending conversation.")
        break




    chat_history.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(
        model="openai/gpt-5-chat-latest",
        messages=chat_history,
        temperature=0.7,
        top_p=0.7
    )
    message = response.choices[0].message.content
    print(f"\nAssistant: {message}\n")
    chat_history.append({"role": "assistant", "content": message})
